In [21]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    #ds_2024["train"],
    ds_2025["train"]
])

all_events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)

Map:   0%|          | 0/5423 [00:00<?, ? examples/s]

In [22]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from pprint import pprint

#embedding_model = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = OllamaEmbeddings(model="mxbai-embed-large")

events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=embedding_model
)

sections_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="sections",
    embedding_function=embedding_model
)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from ollama import ResponseError
import time
import re
import tiktoken

# https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma
def format_time(seconds):
    seconds = int(seconds)
    mins, secs = divmod(seconds, 60)
    hours, mins = divmod(mins, 60)
    
    if hours:
        return f"{hours}h {mins:02d}m {secs:02d}s"
    else:
        return f"{mins:02d}m {secs:02d}s"

llm = ChatOllama(model="llama3.2:3b")

event_documents = []
event_ids = []

section_documents = []
section_metadatas = []
section_ids = []
section_vectors = [] # we pre-compute to ensure section token lengths are limited

max_safe_tokens = 7000

encoding = tiktoken.get_encoding("cl100k_base")



ei = 0
si = 0

print("Processing events and generating summaries of sections...")
start_time = time.time()

# events = all_events.select(range(0, 100))
events = all_events

shrunk_sections_count = 0

for event in events:
    date_prefix = f"{event['month']} {event['day']}, {event['year']}: "
    text = f"{date_prefix}{event['instruction']}\n\n{event['content']}"

    parts = re.split(r'(### .+)', event["response"])

    event_section_ids = []
    current_title = "Introduction"
    current_content = ""

    for part in parts:
        if part.startswith("###"):
            current_title = part.replace("###", "").strip()
            current_title = current_title.replace("**", "").strip()
            
            current_content = ""
        else:
            current_content = part
            current_content = current_content.replace("\n\n#", "").strip()
            current_content = current_content.replace("\n*", "").strip()
            current_content = current_content.replace("**", "").strip()
            current_content = current_content.replace("\n\n---", "").strip()
            current_content = current_content.strip();

            prompt = ChatPromptTemplate.from_template(
                """
                Summarize the following text clearly and concisely into one sentence. Only output the summary and nothing else.

                {current_content}
                """
            )

            chain = prompt | llm | StrOutputParser()

            summary = chain.invoke({"current_content": current_content})

            section_content = f"{current_title}: {current_content}"

            vectors = None
            shrunk = False

            while vectors is None:
                try:
                    vectors = embedding_model.embed_query(section_content)
                except ResponseError as e:
                    if "input length exceeds" in str(e):
                        prompt = ChatPromptTemplate.from_template(
                            """
                            Summarize and make this content 75% it's length. Only output the summary and nothing else.
            
                            {current_content}
                            """
                        )

                        # Shrink the section
                        chain = prompt | llm | StrOutputParser()
                        current_content = chain.invoke({"current_content": current_content})
                        section_content = f"{current_title}: {current_content}"

                        shrunk_sections_count += 1
                        shrunk = True
                    else:
                        raise

            section_vectors.append(vectors)
            section_documents.append(section_content)
            section_metadatas.append({
                "event_id": f"e{ei}",
                "title": current_title,
                "summary": summary,
                "shrunk": f"{shrunk}"
            })
            section_ids.append(f"s{si}")
            
            event_section_ids.append(f"s{si}")

            # if ei < 300:
            #     print("\n\nSection:")
            #     print((si, current_title, current_content, summary)) # print the last element

            si = si + 1


    event_documents.append(Document(
        page_content = text, # this will be converted to document
        metadata={
            "month": event["month"],
            "day": event["day"],
            "year": event["year"],
            "category": event["section"],
            "sections": event_section_ids
        })
    )
    
    event_ids.append(f"e{ei}")
    ei=ei+1

    elapsed = time.time() - start_time
    progress = ei / len(events)

    eta = elapsed/progress * (1-progress)
    
    print(f"\r{ei} / {len(events)} [ {round((ei) / len(events) * 100, 2)}% ] | ETA: {format_time(eta)}", end="")


print(f"\nDone. Shrunk {shrunk_sections_count} sections.")

batch_size = 100

print("\nGenerating embeddings and storing events...")
events_vector_store.reset_collection()

for i in range(0, len(event_documents), batch_size):
    doc_batch = event_documents[i:i+batch_size]
    id_batch = event_ids[i:i+batch_size]
    
    events_vector_store.add_documents(ids=id_batch, documents=doc_batch)

    print(f"\r{i + len(doc_batch)} / {len(event_documents)} [ {round((i + len(doc_batch)) / len(event_documents) * 100)}% ]", end="")

print("\nDone")


print("\nGenerating embeddings and storing sections...")
sections_vector_store.reset_collection()

for i in range(0, len(section_documents), batch_size):
    doc_batch = section_documents[i:i+batch_size]
    meta_batch = section_metadatas[i:i+batch_size]
    id_batch = section_ids[i:i+batch_size]
    vectors_batch = section_vectors[i:i+batch_size]

    # bypassing langchain to write section documents directly to use pre-compued embeddings
    sections_vector_store._collection.add(
        ids=id_batch,
        embeddings=vectors_batch,
        documents=doc_batch,
        metadatas=meta_batch
    )

    print(f"\r{i + len(doc_batch)} / {len(section_documents)} [ {round((i + len(doc_batch)) / len(section_documents) * 100)}% ]", end="")

print("\nDone!")





Processing events and generating summaries of sections...
51 / 5423 [ 0.94% ] | ETA: 6h 36m 45s

In [ ]:
results = events_vector_store.get(    
    limit=3,
    include=["embeddings", "documents", "metadatas"]
)
pprint(results)

results = sections_vector_store.get(
    limit=3,
    include=["embeddings", "documents", "metadatas"]
)
pprint(results)